In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 드라이브 마운트 후 경로 예시
import os

# 이미지에 보이는 구조대로라면 경로가 이렇게 될 거예요.
BASE_PATH = '/content/drive/MyDrive/MyDrive/Stability_Project'

# 파일 존재 확인 테스트
if os.path.exists(BASE_PATH):
    print("경로 연결 성공!")
else:
    print("경로를 확인해주세요.")

경로 연결 성공!


In [ ]:
# 압축 해제 폴더 생성
!mkdir -p /content/train_images
!mkdir -p /content/dev_images
!mkdir -p /content/test_images

# train.zip 압축 해제
train_zip_path = os.path.join(BASE_PATH, 'train.zip')
if not os.path.exists(train_zip_path):
    print(f"❌ 오류: {train_zip_path} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
else:
    !unzip -q "{train_zip_path}" -d /content/train_images
    print("✅ train.zip 압축 해제 완료!")
    print("--- /content/train_images 내용 확인 ---")
    !ls -R /content/train_images
    print("---------------------------------------")

# dev.zip 압축 해제
dev_zip_path = os.path.join(BASE_PATH, 'dev.zip')
if not os.path.exists(dev_zip_path):
    print(f"❌ 오류: {dev_zip_path} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
else:
    !unzip -q "{dev_zip_path}" -d /content/dev_images
    print("✅ dev.zip 압축 해제 완료!")
    print("--- /content/dev_images 내용 확인 ---")
    !ls -R /content/dev_images
    print("-------------------------------------")

# test.zip 압축 해제
test_zip_path = os.path.join(BASE_PATH, 'test.zip')
if not os.path.exists(test_zip_path):
    print(f"❌ 오류: {test_zip_path} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
else:
    !unzip -q "{test_zip_path}" -d /content/test_images
    print("✅ test.zip 압축 해제 완료!")
    print("--- /content/test_images 내용 확인 ---")
    !ls -R /content/test_images
    print("-------------------------------------")

In [ ]:
# 1. 데이터 로드
train_df = pd.read_csv(os.path.join(BASE_PATH, 'train.csv'))
val_df = pd.read_csv(os.path.join(BASE_PATH, 'dev.csv'))

print(f"학습 데이터 개수: {len(train_df)}")
print(f"검증 데이터 개수: {len(val_df)}")

In [ ]:
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample_id = str(self.df.iloc[idx]['id'])
        # 폴더 경로 설정
        folder_path = os.path.join(self.root_dir, sample_id)

        # 2개 뷰 로드
        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)

        # 테스트(추론) 모드일 경우 이미지 리스트만 반환
        if self.is_test:
            return views

        # 학습/검증 모드일 경우 라벨 함께 반환
        label = self.label_map[self.df.iloc[idx]['label']]
        return views, label

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.RandomHorizontalFlip(), # 랜덤 수평 뒤집기 추가
    transforms.RandomRotation(15),     # -15도 ~ +15도 랜덤 회전 추가
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), # 색상 변경 추가
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.406])
])

test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 1. 학습/검증 세트 준비 (is_test=False 설정)
train_dataset = MultiViewDataset(train_df, '/content/train_images/train', train_transform, is_test=False)
val_dataset = MultiViewDataset(val_df, '/content/dev_images/dev', test_transform, is_test=False)

train_loader = DataLoader(train_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

# 2. 테스트 세트 준비 (is_test=True 설정)
test_df = pd.read_csv(os.path.join(BASE_PATH, 'sample_submission.csv'))
test_dataset = MultiViewDataset(test_df, '/content/test_images/test', test_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

In [ ]:
class MultiViewResNet(nn.Module):
    def __init__(self, num_classes=1):
        super(MultiViewResNet, self).__init__()
        self.backbone = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(self.backbone.children())[:-1])

        self.classifier = nn.Sequential(
            nn.Linear(512 * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, views):
        # views: [batch, 3, 224, 224] * 2 images
        f1 = self.feature_extractor(views[0]).view(views[0].size(0), -1)
        f2 = self.feature_extractor(views[1]).view(views[1].size(0), -1)

        combined = torch.cat((f1, f2), dim=1)
        return self.classifier(combined)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    train_loss = 0
    for views, labels in tqdm(loader, desc="Training"):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(views).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    return train_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for views, labels in tqdm(loader, desc="Validation"):
            views = [v.to(device) for v in views]
            labels = labels.to(device).float()

            outputs = model(views).view(-1)
            # 1. 시그모이드를 통과시켜 확률값(unstable일 확률)으로 변환
            probs = torch.sigmoid(outputs)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)

    # 대회 공식 LOGLOSS 계산
    eps = 1e-15
    p = np.clip(all_probs, eps, 1 - eps)
    # Binary Log Loss 공식 직접 적용
    logloss_score = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))

    # Accuracy 계산
    acc_score = np.mean((all_probs > 0.5) == all_labels)

    return logloss_score, acc_score

In [ ]:
model = MultiViewResNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# --- Main Loop ---
for epoch in range(1, CFG['EPOCHS'] + 1):
    avg_train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_logloss, val_acc = validate(model, val_loader, criterion, device)

    # 스케줄러 스텝 (검증 손실 기반)
    scheduler.step(val_logloss)

    print(f"Epoch [{epoch}]")
    print(f"  - Train Loss: {avg_train_loss:.4f}")
    print(f"  - Val Log-Loss: {val_logloss:.6f} | Val Acc: {val_acc:.4f}")

In [ ]:
# 압축 해제 폴더 생성
!mkdir -p /content/train_images
!mkdir -p /content/dev_images
!mkdir -p /content/test_images

# train.zip 압축 해제
train_zip_path = os.path.join(BASE_PATH, 'train.zip')
if not os.path.exists(train_zip_path):
    print(f"❌ 오류: {train_zip_path} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
else:
    !unzip -q "{train_zip_path}" -d /content/train_images
    print("✅ train.zip 압축 해제 완료!")
    print("--- /content/train_images 내용 확인 ---")
    !ls -R /content/train_images
    print("---------------------------------------")

# dev.zip 압축 해제
dev_zip_path = os.path.join(BASE_PATH, 'dev.zip')
if not os.path.exists(dev_zip_path):
    print(f"❌ 오류: {dev_zip_path} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
else:
    !unzip -q "{dev_zip_path}" -d /content/dev_images
    print("✅ dev.zip 압축 해제 완료!")
    print("--- /content/dev_images 내용 확인 ---")
    !ls -R /content/dev_images
    print("-------------------------------------")

# test.zip 압축 해제
test_zip_path = os.path.join(BASE_PATH, 'test.zip')
if not os.path.exists(test_zip_path):
    print(f"❌ 오류: {test_zip_path} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
else:
    !unzip -q "{test_zip_path}" -d /content/test_images
    print("✅ test.zip 압축 해제 완료!")
    print("--- /content/test_images 내용 확인 ---")
    !ls -R /content/test_images
    print("-------------------------------------")

In [ ]:
# 1. 데이터 로드
train_df = pd.read_csv(os.path.join(BASE_PATH, 'train.csv'))
val_df = pd.read_csv(os.path.join(BASE_PATH, 'dev.csv'))

print(f"학습 데이터 개수: {len(train_df)}")
print(f"검증 데이터 개수: {len(val_df)}")

In [ ]:
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample_id = str(self.df.iloc[idx]['id'])
        # 폴더 경로 설정
        folder_path = os.path.join(self.root_dir, sample_id)

        # 2개 뷰 로드
        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)

        # 테스트(추론) 모드일 경우 이미지 리스트만 반환
        if self.is_test:
            return views

        # 학습/검증 모드일 경우 라벨 함께 반환
        label = self.label_map[self.df.iloc[idx]['label']]
        return views, label

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.RandomHorizontalFlip(), # 랜덤 수평 뒤집기 추가
    transforms.RandomRotation(15),     # -15도 ~ +15도 랜덤 회전 추가
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), # 색상 변경 추가
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.406])
])

test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 1. 학습/검증 세트 준비 (is_test=False 설정)
train_dataset = MultiViewDataset(train_df, '/content/train_images/train', train_transform, is_test=False)
val_dataset = MultiViewDataset(val_df, '/content/dev_images/dev', test_transform, is_test=False)

train_loader = DataLoader(train_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

# 2. 테스트 세트 준비 (is_test=True 설정)
test_df = pd.read_csv(os.path.join(BASE_PATH, 'sample_submission.csv'))
test_dataset = MultiViewDataset(test_df, '/content/test_images/test', test_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

In [ ]:
class MultiViewResNet(nn.Module):
    def __init__(self, num_classes=1):
        super(MultiViewResNet, self).__init__()
        self.backbone = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(self.backbone.children())[:-1])

        self.classifier = nn.Sequential(
            nn.Linear(512 * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, views):
        # views: [batch, 3, 224, 224] * 2 images
        f1 = self.feature_extractor(views[0]).view(views[0].size(0), -1)
        f2 = self.feature_extractor(views[1]).view(views[1].size(0), -1)

        combined = torch.cat((f1, f2), dim=1)
        return self.classifier(combined)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    train_loss = 0
    for views, labels in tqdm(loader, desc="Training"):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(views).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    return train_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for views, labels in tqdm(loader, desc="Validation"):
            views = [v.to(device) for v in views]
            labels = labels.to(device).float()

            outputs = model(views).view(-1)
            # 1. 시그모이드를 통과시켜 확률값(unstable일 확률)으로 변환
            probs = torch.sigmoid(outputs)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)

    # 대회 공식 LOGLOSS 계산
    eps = 1e-15
    p = np.clip(all_probs, eps, 1 - eps)
    # Binary Log Loss 공식 직접 적용
    logloss_score = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))

    # Accuracy 계산
    acc_score = np.mean((all_probs > 0.5) == all_labels)

    return logloss_score, acc_score

In [ ]:
model = MultiViewResNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# --- Main Loop ---
for epoch in range(1, CFG['EPOCHS'] + 1):
    avg_train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_logloss, val_acc = validate(model, val_loader, criterion, device)

    # 스케줄러 스텝 (검증 손실 기반)
    scheduler.step(val_logloss)

    print(f"Epoch [{epoch}]")
    print(f"  - Train Loss: {avg_train_loss:.4f}")
    print(f"  - Val Log-Loss: {val_logloss:.6f} | Val Acc: {val_acc:.4f}")

In [25]:
class MultiViewResNet(nn.Module):
    def __init__(self, num_classes=1):
        super(MultiViewResNet, self).__init__()
        self.backbone = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(self.backbone.children())[:-1])

        self.classifier = nn.Sequential(
            nn.Linear(512 * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, views):
        # views: [batch, 3, 224, 224] * 2 images
        f1 = self.feature_extractor(views[0]).view(views[0].size(0), -1)
        f2 = self.feature_extractor(views[1]).view(views[1].size(0), -1)

        combined = torch.cat((f1, f2), dim=1)
        return self.classifier(combined)

In [ ]:
model = MultiViewResNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# --- Main Loop ---
for epoch in range(1, CFG['EPOCHS'] + 1):
    avg_train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_logloss, val_acc = validate(model, val_loader, criterion, device)

    # 스케줄러 스텝 (검증 손실 기반)
    scheduler.step(val_logloss)

    print(f"Epoch [{epoch}]")
    print(f"  - Train Loss: {avg_train_loss:.4f}")
    print(f"  - Val Log-Loss: {val_logloss:.6f} | Val Acc: {val_acc:.4f}")

Training:   0%|          | 0/32 [00:00<?, ?it/s]

## 1. 라이브러리 및 환경 설정

In [3]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# 하이퍼파라미터 설정
CFG = {
    'IMG_SIZE': 224,
    'EPOCHS': 20, # 에포크 수를 10으로 변경
    'LEARNING_RATE': 1e-3,
    'BATCH_SIZE': 32,
    'SEED': 42
}

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. 데이터 로드 및 학습/검증 데이터 분할

In [22]:
# 1. 데이터 로드
train_df = pd.read_csv(os.path.join(BASE_PATH, 'train.csv'))
val_df = pd.read_csv(os.path.join(BASE_PATH, 'dev.csv'))

print(f"학습 데이터 개수: {len(train_df)}")
print(f"검증 데이터 개수: {len(val_df)}")

학습 데이터 개수: 1000
검증 데이터 개수: 100


## 3. 커스텀 데이터셋 클래스 정의 및 데이터 로더

In [23]:
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample_id = str(self.df.iloc[idx]['id'])
        # 폴더 경로 설정
        folder_path = os.path.join(self.root_dir, sample_id)

        # 2개 뷰 로드
        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)

        # 테스트(추론) 모드일 경우 이미지 리스트만 반환
        if self.is_test:
            return views

        # 학습/검증 모드일 경우 라벨 함께 반환
        label = self.label_map[self.df.iloc[idx]['label']]
        return views, label

In [24]:
train_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.RandomHorizontalFlip(), # 랜덤 수평 뒤집기 추가
    transforms.RandomRotation(15),     # -15도 ~ +15도 랜덤 회전 추가
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), # 색상 변경 추가
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.406])
])

test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 1. 학습/검증 세트 준비 (is_test=False 설정)
train_dataset = MultiViewDataset(train_df, '/content/train_images/train', train_transform, is_test=False)
val_dataset = MultiViewDataset(val_df, '/content/dev_images/dev', test_transform, is_test=False)

train_loader = DataLoader(train_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

# 2. 테스트 세트 준비 (is_test=True 설정)
test_df = pd.read_csv(os.path.join(BASE_PATH, 'sample_submission.csv'))
test_dataset = MultiViewDataset(test_df, '/content/test_images/test', test_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

## 4. 모델 정의 (Multi-View )

In [5]:
class MultiViewResNet(nn.Module):
    def __init__(self, num_classes=1):
        super(MultiViewResNet, self).__init__()
        self.backbone = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(self.backbone.children())[:-1])

        self.classifier = nn.Sequential(
            nn.Linear(512 * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, views):
        # views: [batch, 3, 224, 224] * 2 images
        f1 = self.feature_extractor(views[0]).view(views[0].size(0), -1)
        f2 = self.feature_extractor(views[1]).view(views[1].size(0), -1)

        combined = torch.cat((f1, f2), dim=1)
        return self.classifier(combined)

## 4. 학습 및 검증 루프 구현

In [26]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    train_loss = 0
    for views, labels in tqdm(loader, desc="Training"):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(views).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    return train_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for views, labels in tqdm(loader, desc="Validation"):
            views = [v.to(device) for v in views]
            labels = labels.to(device).float()

            outputs = model(views).view(-1)
            # 1. 시그모이드를 통과시켜 확률값(unstable일 확률)으로 변환
            probs = torch.sigmoid(outputs)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)

    # 대회 공식 LOGLOSS 계산
    eps = 1e-15
    p = np.clip(all_probs, eps, 1 - eps)
    # Binary Log Loss 공식 직접 적용
    logloss_score = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))

    # Accuracy 계산
    acc_score = np.mean((all_probs > 0.5) == all_labels)

    return logloss_score, acc_score

In [17]:
model = MultiViewResNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# --- Main Loop ---
for epoch in range(1, CFG['EPOCHS'] + 1):
    avg_train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_logloss, val_acc = validate(model, val_loader, criterion, device)

    # 스케줄러 스텝 (검증 손실 기반)
    scheduler.step(val_logloss)

    print(f"Epoch [{epoch}]")
    print(f"  - Train Loss: {avg_train_loss:.4f}")
    print(f"  - Val Log-Loss: {val_logloss:.6f} | Val Acc: {val_acc:.4f}")

NameError: name 'train_loader' is not defined

In [ ]:
# /content/train_images 디렉토리의 내용을 확인하여 실제 파일 구조를 파악합니다.
!ls -R /content/train_images

/content/train_images:
train

/content/train_images/train:
TRAIN_0001  TRAIN_0168	TRAIN_0335  TRAIN_0502	TRAIN_0669  TRAIN_0836
TRAIN_0002  TRAIN_0169	TRAIN_0336  TRAIN_0503	TRAIN_0670  TRAIN_0837
TRAIN_0003  TRAIN_0170	TRAIN_0337  TRAIN_0504	TRAIN_0671  TRAIN_0838
TRAIN_0004  TRAIN_0171	TRAIN_0338  TRAIN_0505	TRAIN_0672  TRAIN_0839
TRAIN_0005  TRAIN_0172	TRAIN_0339  TRAIN_0506	TRAIN_0673  TRAIN_0840
TRAIN_0006  TRAIN_0173	TRAIN_0340  TRAIN_0507	TRAIN_0674  TRAIN_0841
TRAIN_0007  TRAIN_0174	TRAIN_0341  TRAIN_0508	TRAIN_0675  TRAIN_0842
TRAIN_0008  TRAIN_0175	TRAIN_0342  TRAIN_0509	TRAIN_0676  TRAIN_0843
TRAIN_0009  TRAIN_0176	TRAIN_0343  TRAIN_0510	TRAIN_0677  TRAIN_0844
TRAIN_0010  TRAIN_0177	TRAIN_0344  TRAIN_0511	TRAIN_0678  TRAIN_0845
TRAIN_0011  TRAIN_0178	TRAIN_0345  TRAIN_0512	TRAIN_0679  TRAIN_0846
TRAIN_0012  TRAIN_0179	TRAIN_0346  TRAIN_0513	TRAIN_0680  TRAIN_0847
TRAIN_0013  TRAIN_0180	TRAIN_0347  TRAIN_0514	TRAIN_0681  TRAIN_0848
TRAIN_0014  TRAIN_0181	TRAIN_0348  TRAIN_051

## 5. 추론 및 제출 파일 생성

In [21]:
# 압축 해제 폴더 생성
!mkdir -p /content/train_images
!mkdir -p /content/dev_images
!mkdir -p /content/test_images

# train.zip 압축 해제
train_zip_path = os.path.join(BASE_PATH, 'train.zip')
if not os.path.exists(train_zip_path):
    print(f"❌ 오류: {train_zip_path} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
else:
    !unzip -q "{train_zip_path}" -d /content/train_images
    print("✅ train.zip 압축 해제 완료!")
    print("--- /content/train_images 내용 확인 ---")
    !ls -R /content/train_images
    print("---------------------------------------")

# dev.zip 압축 해제
dev_zip_path = os.path.join(BASE_PATH, 'dev.zip')
if not os.path.exists(dev_zip_path):
    print(f"❌ 오류: {dev_zip_path} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
else:
    !unzip -q "{dev_zip_path}" -d /content/dev_images
    print("✅ dev.zip 압축 해제 완료!")
    print("--- /content/dev_images 내용 확인 ---")
    !ls -R /content/dev_images
    print("-------------------------------------")

# test.zip 압축 해제
test_zip_path = os.path.join(BASE_PATH, 'test.zip')
if not os.path.exists(test_zip_path):
    print(f"❌ 오류: {test_zip_path} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
else:
    !unzip -q "{test_zip_path}" -d /content/test_images
    print("✅ test.zip 압축 해제 완료!")
    print("--- /content/test_images 내용 확인 ---")
    !ls -R /content/test_images
    print("-------------------------------------")

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.

/content/train_images/train/TRAIN_0491:
front.png  simulation.mp4  top.png

/content/train_images/train/TRAIN_0492:
front.png  simulation.mp4  top.png

/content/train_images/train/TRAIN_0493:
front.png  simulation.mp4  top.png

/content/train_images/train/TRAIN_0494:
front.png  simulation.mp4  top.png

/content/train_images/train/TRAIN_0495:
front.png  simulation.mp4  top.png

/content/train_images/train/TRAIN_0496:
front.png  simulation.mp4  top.png

/content/train_images/train/TRAIN_0497:
front.png  simulation.mp4  top.png

/content/train_images/train/TRAIN_0498:
front.png  simulation.mp4  top.png

/content/train_images/train/TRAIN_0499:
front.png  simulation.mp4  top.png

/content/train_images/train/TRAIN_0500:
front.png  simulation.mp4  top.png

/content/train_images/train/TRAIN_0501:
front.png  simulation.mp4  top.png

/content/train_images/train/TRAIN_0502:
front.png  simulation.mp4  top.png

/content/train_images/train/TRAIN_0503:
front.png  

In [ ]:
model.eval()
all_probs = []

with torch.no_grad():
    for views in tqdm(test_loader, desc="Inference"):
        views = [v.to(device) for v in views]

        # 모델 출력 (Logit) -> 시그모이드 -> unstable(1)일 확률
        outputs = model(views).view(-1)
        probs = torch.sigmoid(outputs)

        all_probs.extend(probs.cpu().numpy())

all_probs = np.array(all_probs)
# 결과 저장 (컬럼  순서 중요)
submission = pd.DataFrame({
    'id': test_df['id'],
    'unstable_prob': all_probs,  # unstable일 확률 저장
    'stable_prob': 1.0 - all_probs # stable일 확률 저장
})

submission.to_csv('submission.csv', encoding='UTF-8-sig', index=False)
print("submission.csv 저장 완료.")

Inference:   0%|          | 0/32 [00:00<?, ?it/s]

submission.csv 저장 완료.


In [ ]:
# /content/test_images 디렉토리의 내용을 확인하여 실제 파일 구조를 파악합니다.
!ls -R /content/test_images

/content/test_images:
test

/content/test_images/test:
TEST_0001  TEST_0144  TEST_0287  TEST_0430  TEST_0573  TEST_0716  TEST_0859
TEST_0002  TEST_0145  TEST_0288  TEST_0431  TEST_0574  TEST_0717  TEST_0860
TEST_0003  TEST_0146  TEST_0289  TEST_0432  TEST_0575  TEST_0718  TEST_0861
TEST_0004  TEST_0147  TEST_0290  TEST_0433  TEST_0576  TEST_0719  TEST_0862
TEST_0005  TEST_0148  TEST_0291  TEST_0434  TEST_0577  TEST_0720  TEST_0863
TEST_0006  TEST_0149  TEST_0292  TEST_0435  TEST_0578  TEST_0721  TEST_0864
TEST_0007  TEST_0150  TEST_0293  TEST_0436  TEST_0579  TEST_0722  TEST_0865
TEST_0008  TEST_0151  TEST_0294  TEST_0437  TEST_0580  TEST_0723  TEST_0866
TEST_0009  TEST_0152  TEST_0295  TEST_0438  TEST_0581  TEST_0724  TEST_0867
TEST_0010  TEST_0153  TEST_0296  TEST_0439  TEST_0582  TEST_0725  TEST_0868
TEST_0011  TEST_0154  TEST_0297  TEST_0440  TEST_0583  TEST_0726  TEST_0869
TEST_0012  TEST_0155  TEST_0298  TEST_0441  TEST_0584  TEST_0727  TEST_0870
TEST_0013  TEST_0156  TEST_0299  